# Assignment 3: Network Biology

**Group:** Group 02  
**Members:** Claire Bams, Yustyna Babichuk, Antonia Constantin, Francisco Javier Camacho Perez de Sevilla

This notebook compares the normal Boolean regulatory network with four mutated versions. Each network will be tested with the same scenarios, attractor analysis, and basin-size analysis.

## 1. Setup

Run this section first. The imports and helper code below are shared by every group member.

In [1]:
import numpy as np
import pandas as pd
from itertools import product
from collections import defaultdict

## 2. Boolean network model

The class below simulates synchronous Boolean-network updates. `create_normal_network()` returns a fresh copy of the normal eight-node network, so every mutation starts from the same baseline.

In [2]:
# This class stores the node states and Boolean update rules.
class BooleanNetwork:
    def __init__(self, node_names):
        # Every node starts OFF (0). A scenario will set the starting values later.
        self.nodes = {name: 0 for name in node_names}
        self.rules = {}
        self.history = []

    def add_rule(self, target_node, rule_function, rule_description=""):
        # Save both the executable rule and a readable description of it.
        self.rules[target_node] = {
            'function': rule_function,
            'description': rule_description
        }

    def set_state(self, **kwargs):
        # Set the requested node values and convert True/False to 1/0.
        for node, value in kwargs.items():
            if node in self.nodes:
                self.nodes[node] = int(bool(value))

    def get_state_vector(self):
        # Always use alphabetical node order so results can be compared safely.
        return [self.nodes[node] for node in sorted(self.nodes.keys())]

    def update_synchronous(self):
        # Calculate every new value from the same current state.
        new_state = {}
        for node in self.nodes:
            if node in self.rules:
                new_state[node] = int(self.rules[node]['function'](self.nodes))
            else:
                new_state[node] = self.nodes[node]

        self.nodes = new_state
        self.history.append(self.get_state_vector())

    def simulate(self, steps=10, record_history=True, verbose=False):
        # Store the starting state, then update until stable or out of steps.
        if record_history:
            self.history = [self.get_state_vector()]

        for step in range(steps):
            self.update_synchronous()

            # Two identical consecutive states mean a fixed point was reached.
            if len(self.history) >= 2 and self.history[-1] == self.history[-2]:
                if verbose:
                    print(f"Reached steady state after {step + 1} steps")
                break

        return np.array(self.history)

In [3]:
# Build a new, independent copy of the normal eight-node network.
# Always call this function before applying a mutation.
def create_normal_network():
    nodes = [
        'DNA_damage', 'p53', 'MYC', 'CDK2',
        'MDM2', 'p21', 'Growth', 'Death'
    ]

    network = BooleanNetwork(nodes)

    # Normal Boolean rules copied from the completed practical.
    network.add_rule('DNA_damage', lambda s: s['DNA_damage'], "DNA_damage = INPUT (constant)")
    network.add_rule('p21', lambda s: s['p53'], "p21 = p53")
    network.add_rule('MYC', lambda s: (not s['p53']) and (not s['p21']), "MYC = (NOT p53) AND (NOT p21)")
    network.add_rule('CDK2', lambda s: s['MYC'] and (not s['p21']) and (not s['p53']), "CDK2 = MYC AND (NOT p21) AND (NOT p53)")
    network.add_rule('MDM2', lambda s: s['MYC'], "MDM2 = MYC")
    network.add_rule('p53', lambda s: s['DNA_damage'] and (not s['MDM2']), "p53 = DNA_damage AND (NOT MDM2)")
    network.add_rule('Growth', lambda s: s['CDK2'] and s['MYC'] and (not s['p53']), "Growth = CDK2 AND MYC AND (NOT p53)")
    network.add_rule('Death', lambda s: s['p53'] and s['DNA_damage'] and (not s['Growth']), "Death = p53 AND DNA_damage AND (NOT Growth)")

    return network

## 3. Shared analysis functions

These functions ensure that the normal network and all four mutations are analysed in exactly the same way. Group members should use them instead of copying the practical code into every section.

This practical searches for **fixed-point attractors**: states that no longer change. It does not search for longer repeating cycles.

In [4]:
# Starting states required by the practical and reused for every network.
SCENARIOS = {
    'Healthy Cell': {
        'DNA_damage': 0, 'p53': 0, 'MYC': 0, 'CDK2': 0,
        'MDM2': 0, 'p21': 0, 'Growth': 0, 'Death': 0
    },
    'Stressed Cell': {
        'DNA_damage': 1, 'p53': 0, 'MYC': 0, 'CDK2': 0,
        'MDM2': 0, 'p21': 0, 'Growth': 0, 'Death': 0
    },
    'Oncogene Hijacked Cell': {
        'DNA_damage': 0, 'p53': 0, 'MYC': 1, 'CDK2': 0,
        'MDM2': 0, 'p21': 0, 'Growth': 0, 'Death': 0
    }
}


# Run all three scenarios and collect the required final outputs.
def run_scenarios(network, steps=15):
    rows = []
    trajectories = {}
    node_names = sorted(network.nodes.keys())

    for scenario_name, initial_state in SCENARIOS.items():
        # Reset the network to this scenario before starting the simulation.
        network.set_state(**initial_state)
        trajectory = network.simulate(steps=steps)
        trajectories[scenario_name] = trajectory

        # Convert the final state vector back into named node values.
        final_state = {
            node: int(trajectory[-1][i])
            for i, node in enumerate(node_names)
        }

        # The assignment specifically asks for Growth, Death, and p53.
        rows.append({
            'Scenario': scenario_name,
            'Growth': final_state['Growth'],
            'Death': final_state['Death'],
            'p53': final_state['p53']
        })

    return pd.DataFrame(rows), trajectories

In [5]:
# Test all 2^8 starting states and collect unique fixed-point attractors.
def find_attractors(network, max_steps=15):
    attractors = []
    node_names = sorted(network.nodes.keys())

    # product([0, 1], repeat=8) generates all 256 possible starting states.
    for initial_state in product([0, 1], repeat=len(node_names)):
        network.set_state(**dict(zip(node_names, initial_state)))
        trajectory = network.simulate(steps=max_steps)

        # A fixed point has identical states in the final two time steps.
        reached_fixed_point = (
            len(trajectory) >= 2
            and np.array_equal(trajectory[-1], trajectory[-2])
        )

        if reached_fixed_point:
            final_state = tuple(int(x) for x in trajectory[-1])
            if final_state not in attractors:
                attractors.append(final_state)

    return attractors


# Give each attractor a consistent biological interpretation.
def classify_attractor(attractor, node_names):
    state = dict(zip(node_names, attractor))

    if state['Growth'] == 1 and state['Death'] == 0 and state['DNA_damage'] == 0:
        return 'Healthy growth'
    if state['Death'] == 1 and state['Growth'] == 0:
        return 'Cell death'
    if state['Growth'] == 1 and state['Death'] == 0 and state['DNA_damage'] == 1:
        return 'Cancer-like growth'
    return 'Other/conflicting state'

In [6]:
# Count how many of the 256 initial states lead to each attractor.
def calculate_basins(network, attractors, max_steps=15):
    node_names = sorted(network.nodes.keys())
    basin_data = defaultdict(list)

    # This lookup connects each final state to its attractor number.
    attractor_lookup = {
        tuple(int(x) for x in attractor): index
        for index, attractor in enumerate(attractors)
    }

    all_states = list(product([0, 1], repeat=len(node_names)))

    for initial_state in all_states:
        network.set_state(**dict(zip(node_names, initial_state)))
        trajectory = network.simulate(steps=max_steps)

        reached_fixed_point = (
            len(trajectory) >= 2
            and np.array_equal(trajectory[-1], trajectory[-2])
        )

        if reached_fixed_point:
            final_state = tuple(int(x) for x in trajectory[-1])
            if final_state in attractor_lookup:
                attractor_index = attractor_lookup[final_state]
                basin_data[attractor_index].append(initial_state)

    # Make one readable row for every attractor.
    rows = []
    for index, attractor in enumerate(attractors):
        basin_size = len(basin_data[index])
        rows.append({
            'Attractor': index + 1,
            'State': tuple(int(x) for x in attractor),
            'Classification': classify_attractor(attractor, node_names),
            'Basin size': basin_size,
            'Basin percentage': 100 * basin_size / len(all_states)
        })

    return pd.DataFrame(rows)


# Run the complete required analysis for one normal or mutated network.
def analyse_network(network):
    scenario_results, trajectories = run_scenarios(network)
    attractors = find_attractors(network)
    basin_results = calculate_basins(network, attractors)

    return {
        'scenarios': scenario_results,
        'trajectories': trajectories,
        'attractors': attractors,
        'basins': basin_results
    }

## 4. Normal-network baseline


### TODO

- Run the shared analysis on the unmodified network.
- Record the three scenario outcomes.
- Record the attractors, classifications, basin sizes, and percentages.
- Keep these results as the reference for all mutations.

In [7]:
# Create and analyse the normal network
normal_network = create_normal_network()
normal_results = analyse_network(normal_network)

# Show the final Growth, Death, and p53 values for all three scenarios
print("NORMAL NETWORK — SCENARIO ANALYSIS")
display(normal_results['scenarios'])

# Show every attractor, its classification, basin size, and percentage
print("\nNORMAL NETWORK — ATTRACTOR AND BASIN ANALYSIS")
display(normal_results['basins'])

# Count the number of attractors
number_of_attractors = len(normal_results['attractors'])
print(f"\nNumber of attractors: {number_of_attractors}")

# Select all attractors classified as cancer-like
cancer_like_attractors = normal_results['basins'][
    normal_results['basins']['Classification'] == 'Cancer-like growth'
]

# Add their basin sizes and percentages
cancer_like_basin_size = cancer_like_attractors['Basin size'].sum()
cancer_like_percentage = cancer_like_attractors['Basin percentage'].sum()

print(f"Cancer-like basin size: {cancer_like_basin_size} out of 256 states")
print(f"Percentage leading to cancer-like growth: {cancer_like_percentage:.1f}%")

NORMAL NETWORK — SCENARIO ANALYSIS


,Scenario,Growth,Death,p53
0,Healthy Cell,1,0,0
1,Stressed Cell,0,1,1
2,Oncogene Hijacked Cell,1,0,0



NORMAL NETWORK — ATTRACTOR AND BASIN ANALYSIS


,Attractor,State,Classification,Basin size,Basin percentage
0,1,"(1, 0, 0, 1, 1, 1, 0, 0)",Healthy growth,128,50.000
1,2,"(0, 1, 1, 0, 0, 0, 1, 1)",Cell death,120,46.875
2,3,"(1, 1, 0, 1, 1, 1, 0, 0)",Cancer-like growth,8,3.125



Number of attractors: 3
Cancer-like basin size: 8 out of 256 states
Percentage leading to cancer-like growth: 3.1%


## 5. Mutation A — p53 knockout


### TODO

- Start with a fresh normal network.
- Override the `p53` rule so that p53 is always OFF.
- Run the shared analysis.
- Record the scenario outcomes and basin results.
- Calculate the percentage of states leading to cancer-like growth.
- Add a short interpretation.

In [12]:
mutation_a = create_normal_network()

# TODO: Remove the # from the next line to apply the required p53 knockout.
mutation_a.add_rule('p53', lambda s: False, "p53 = BROKEN (always OFF)")

# TODO: Remove the # from these lines to run and display the analysis.
mutation_a_results = analyse_network(mutation_a)
mutation_a_results['scenarios']
mutation_a_results['basins']

,Attractor,State,Classification,Basin size,Basin percentage
0,1,"(1, 0, 0, 1, 1, 1, 0, 0)",Healthy growth,128,50.0
1,2,"(1, 1, 0, 1, 1, 1, 0, 0)",Cancer-like growth,128,50.0


### Interpretation:
As seen in the table, the percentage of states leadin to cancer-like growth is exactly 50.0% out of all 256.
Inactivating p53 eliminates the ability of the cell to trigger apoptpsis in response to DNA damage. p1 can no longer activate nor can it trigger the Death node. Consequently, the protective "Cell death" attractor  disappears and every state that would normally result in programmed cell death now stabilizes into uncontrolled, cancer-like growth, which increases from 3.1% to 50% of all possible network states.

## 6. Mutation B — MYC amplification


### TODO

- Start with a fresh normal network.
- Override the `MYC` rule so that MYC is always ON.
- Run the shared analysis.
- Record the scenario outcomes and basin results.
- Calculate the percentage of states leading to cancer-like growth.
- Add a short interpretation.

In [15]:
mutation_b = create_normal_network()

# TODO: Remove the # from the next line to apply the required MYC amplification.
mutation_b.add_rule('MYC', lambda s: True, "MYC = AMPLIFIED (always ON)")

# TODO: Remove the # from these lines to run and display the analysis.
mutation_b_results = analyse_network(mutation_b)
mutation_b_results['scenarios']
mutation_b_results['basins']

,Attractor,State,Classification,Basin size,Basin percentage
0,1,"(1, 0, 0, 1, 1, 1, 0, 0)",Healthy growth,128,50.0
1,2,"(1, 1, 0, 1, 1, 1, 0, 0)",Cancer-like growth,128,50.0


### Interpretation:
Like with the p53 inactivation, the percentage of states leading to the cancer-like growth attractor is 50.0%.
Amplifying the MYC oncogene forces it to remain constantly active. This permanent MYC activity drives the overexpression of MDM2, which in turn  degrades p53. Because p53 is suppressed entirely by the excess MDM2, the cell is blind to DNA damage and it loses its ability to trigger apoptosis, making 50% of all possible initial states to stabilize into cancer-like growth instead of cell death.

## 7. Mutation C — MDM2 overexpression


### TODO

- Start with a fresh normal network.
- Override the `MDM2` rule so that MDM2 is always ON.
- Run the shared analysis.
- Record the scenario outcomes and basin results.
- Calculate the percentage of states leading to cancer-like growth.
- Add a short interpretation.

In [10]:
mutation_c = create_normal_network()

# TODO: Remove the # from the next line to apply the required MDM2 overexpression.
# mutation_c.add_rule('MDM2', lambda s: True, "MDM2 = OVEREXPRESSED (always ON)")

# TODO: Remove the # from these lines to run and display the analysis.
# mutation_c_results = analyse_network(mutation_c)
# mutation_c_results['scenarios']
# mutation_c_results['basins']

## 8. Mutation D — group-selected mutation


### TODO

- Choose one biologically meaningful mutation as a group.
- Give the mutation a clear name and explain why it was selected.
- Start with a fresh normal network and override the chosen rule.
- Run the same scenario, attractor, and basin analyses.
- Calculate the percentage of states leading to cancer-like growth.
- Add a short interpretation.

In [11]:
mutation_d = create_normal_network()

# TODO: Add the group-selected mutation rule here.
# Follow the same pattern as Mutations A-C, but change the chosen node and rule.

# TODO: Remove the # from these lines after adding the mutation rule.
# mutation_d_results = analyse_network(mutation_d)
# mutation_d_results['scenarios']
# mutation_d_results['basins']

## 9. Comparison of all networks


### TODO

- Combine the normal-network and mutation results into one comparison table.
- Compare the cancer-like basin percentages.
- Decide which mutation is most dangerous using quantitative evidence.
- Explain the role of relevant feedback loops.
- Discuss three specific limitations of this Boolean model.
- Add one comparison figure only if it makes the results easier to understand.

| Network | Cancer-like basin size | Cancer-like percentage |
|---|---:|---:|
| Normal network | 8 | 3.1% |
| p53 knockout | TODO | TODO |
| MYC amplification | TODO | TODO |
| MDM2 overexpression | TODO | TODO |
| Mutation D | TODO | TODO |

## 10. Conclusion

### TODO

After all analyses are complete, write one short paragraph summarizing the main differences between the normal network and the four mutations.